# NordikBank AML: Feature Engineering

# 1. Exploratory Data Analysis (EDA)

Before engineering any features, we explore the raw data to understand its structure,
quality, and distributions. This ensures that all feature engineering decisions are 
informed by the actual data rather than assumptions.
## 1.1 Setup & Data Loading

We begin by importing all required libraries and loading the six case datasets.
All tables are read from the shared `data/` directory. We confirm row and column 
counts for each table to verify successful loading before any analysis begins.

In [3]:
import pandas as pd

# Load all tables
customers = pd.read_csv('../../data/customers.csv')
accounts = pd.read_csv('../../data/accounts.csv')
transactions = pd.read_csv('../../data/transactions.csv', parse_dates=['timestamp'])
baselines = pd.read_csv('../../data/baselines.csv')
alerts = pd.read_csv('../../data/alert_history.csv')
country_risk = pd.read_csv('../../data/country_risk.csv')

# Quick shape check
for name, df in [('customers', customers), ('accounts', accounts), 
                 ('transactions', transactions), ('baselines', baselines),
                 ('alerts', alerts), ('country_risk', country_risk)]:
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} cols")

customers: 1200 rows, 16 cols
accounts: 2702 rows, 8 cols
transactions: 77384 rows, 15 cols
baselines: 1200 rows, 10 cols
alerts: 19469 rows, 6 cols
country_risk: 48 rows, 4 cols


## 1.2. Target Distribution

We check the class balance of the target variable `suspicious_activity_confirmed` 
across train and val splits. Since suspicious customers are rare by nature, 
understanding the base rate is essential - a high class imbalance risks the model 
defaulting to always predicting "clean", making it useless for detection.

In [4]:
# Filter to labeled rows only (train and val)
labeled = customers[customers['split'].isin(['train', 'val'])]

# Overall target distribution
print("=== Overall target distribution (train + val) ===")
print(labeled['suspicious_activity_confirmed'].value_counts())
print(f"\nBase rate: {labeled['suspicious_activity_confirmed'].mean():.1%}")

# By split
print("\n=== By split ===")
print(labeled.groupby('split')['suspicious_activity_confirmed'].agg(['sum', 'count', 'mean']).rename(columns={'sum': 'suspicious', 'count': 'total', 'mean': 'base_rate'}))

=== Overall target distribution (train + val) ===
suspicious_activity_confirmed
0.0    670
1.0     30
Name: count, dtype: int64

Base rate: 4.3%

=== By split ===
       suspicious  total  base_rate
split                              
train        21.0    500      0.042
val           9.0    200      0.045


## 1.3. Null Profiling

We check null rates across all six tables to identify missing data patterns.
Some nulls are expected by design (e.g. age and income for corporate customers),
while unexpected nulls could silently corrupt features, leading to a model 
trained on incomplete or misleading information.

In [5]:
print("=== Null rates by table ===\n")
for name, df in [('customers', customers), ('accounts', accounts),
                 ('transactions', transactions), ('baselines', baselines),
                 ('alerts', alerts), ('country_risk', country_risk)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) == 0:
        print(f"{name}: no nulls\n")
    else:
        print(f"{name}:")
        for col, n in nulls.items():
            print(f"  {col}: {n} nulls ({n/len(df):.1%})")
        print()

=== Null rates by table ===

customers:
  age: 104 nulls (8.7%)
  occupation_category: 100 nulls (8.3%)
  declared_annual_income: 104 nulls (8.7%)
  declared_annual_turnover: 922 nulls (76.8%)
  industry_code: 922 nulls (76.8%)
  suspicious_activity_confirmed: 500 nulls (41.7%)

accounts: no nulls

transactions:
  counterparty_id: 22808 nulls (29.5%)
  counterparty_bank_country: 74814 nulls (96.7%)
  counterparty_name_hash: 22808 nulls (29.5%)
  merchant_category_code: 62327 nulls (80.5%)

baselines: no nulls

alerts: no nulls

country_risk: no nulls



## 1.4. Customer Type Breakdown

We profile the four customer types (personal, corporate, sole_trader, SME) and 
their suspicious activity rates. Different customer types have fundamentally 
different transaction behaviors and risk profiles, so understanding their 
distribution ensures we never mistakenly treat them as the same during feature engineering.

In [6]:
print("=== Customer type distribution ===")
print(customers['customer_type'].value_counts())

print("\n=== Suspicious rate by customer type (train + val only) ===")
labeled = customers[customers['split'].isin(['train', 'val'])]
print(labeled.groupby('customer_type')['suspicious_activity_confirmed']
      .agg(['sum', 'count', 'mean'])
      .rename(columns={'sum': 'suspicious', 'count': 'total', 'mean': 'base_rate'})
      .round(3))

=== Customer type distribution ===
customer_type
personal       864
sole_trader    147
corporate      100
SME             89
Name: count, dtype: int64

=== Suspicious rate by customer type (train + val only) ===
               suspicious  total  base_rate
customer_type                              
SME                   2.0     48      0.042
corporate             8.0     61      0.131
personal             13.0    499      0.026
sole_trader           7.0     92      0.076


## 1.5 Transaction Profiling

We examine the distribution of transaction types, channels, statuses, and currencies.
This reveals which transaction categories dominate the dataset and flags any 
unusual patterns, such as high declined rates, that are relevant AML signals 
and will inform which features we engineer.

In [7]:
print("=== Transaction types ===")
print(transactions['transaction_type'].value_counts())

print("\n=== Channels ===")
print(transactions['channel'].value_counts())

print("\n=== Status ===")
print(transactions['status'].value_counts())

print("\n=== Currencies ===")
print(transactions['currency'].value_counts())

print("\n=== Amount range (approved only) ===")
approved = transactions[transactions['status'] == 'approved']
print(approved['amount'].describe().round(2))

=== Transaction types ===
transaction_type
transfer              22995
direct_debit          20840
card_payment          15057
standing_order         8264
cash_deposit           4603
cash_withdrawal        3195
international_wire     2430
Name: count, dtype: int64

=== Channels ===
channel
online_banking    57197
mobile_app        12386
branch             4425
ATM                3376
Name: count, dtype: int64

=== Status ===
status
approved    75807
declined     1577
Name: count, dtype: int64

=== Currencies ===
currency
DKK    76357
USD      536
EUR      491
Name: count, dtype: int64

=== Amount range (approved only) ===
count      75807.00
mean       45412.62
std       307451.32
min     -2925988.89
25%         -678.88
50%         -247.46
75%         7600.00
max      7328554.68
Name: amount, dtype: float64


## 1.6 Join Sanity Checks

We verify that all foreign keys across tables are consistent, ensuring every 
transaction and account links back to a valid customer. Broken joins would mean 
we silently lose data during feature engineering, leading to incomplete customer profiles.

In [8]:
# Check all transaction customer_ids exist in customers
txn_customers = set(transactions['customer_id'].unique())
all_customers = set(customers['customer_id'].unique())

missing_in_customers = txn_customers - all_customers
print(f"Transaction customer_ids not in customers: {len(missing_in_customers)}")

# Check all account customer_ids exist in customers
acc_customers = set(accounts['customer_id'].unique())
missing_acc = acc_customers - all_customers
print(f"Account customer_ids not in customers: {len(missing_acc)}")

# Check all alert customer_ids exist in customers
alert_customers = set(alerts['customer_id'].unique())
missing_alerts = alert_customers - all_customers
print(f"Alert customer_ids not in customers: {len(missing_alerts)}")

# Check all baseline customer_ids exist in customers
base_customers = set(baselines['customer_id'].unique())
missing_base = base_customers - all_customers
print(f"Baseline customer_ids not in customers: {len(missing_base)}")

# Check customers with no transactions
no_txn = all_customers - txn_customers
print(f"\nCustomers with no transactions: {len(no_txn)}")

Transaction customer_ids not in customers: 0
Account customer_ids not in customers: 0
Alert customer_ids not in customers: 0
Baseline customer_ids not in customers: 0

Customers with no transactions: 0


## 1.7 Time Coverage

We verify that transactions span the expected Jan–Dec 2024 window and check 
whether coverage is uniform across customers. Gaps in transaction history for 
specific customers could indicate dormancy, itself an AML signal, or data 
quality issues that affect feature reliability.

In [9]:
print("=== Overall date range ===")
print(f"Min: {transactions['timestamp'].min()}")
print(f"Max: {transactions['timestamp'].max()}")

print("\n=== Transactions per month ===")
transactions['month'] = transactions['timestamp'].dt.to_period('M')
print(transactions.groupby('month').size())

print("\n=== Transactions per customer (approved only) ===")
approved = transactions[transactions['status'] == 'approved']
txn_per_customer = approved.groupby('customer_id').size()
print(txn_per_customer.describe().round(2))

print(f"\nCustomers with fewer than 5 transactions: {(txn_per_customer < 5).sum()}")

=== Overall date range ===
Min: 2024-01-01 00:00:00
Max: 2024-12-31 22:57:00

=== Transactions per month ===
month
2024-01    6170
2024-02    6424
2024-03    6245
2024-04    6357
2024-05    6424
2024-06    6334
2024-07    6317
2024-08    6319
2024-09    6486
2024-10    6607
2024-11    6668
2024-12    7033
Freq: M, dtype: int64

=== Transactions per customer (approved only) ===
count    1200.00
mean       63.17
std        28.10
min        18.00
25%        47.00
50%        58.00
75%        70.00
max       206.00
dtype: float64

Customers with fewer than 5 transactions: 0


## 1.8 Country Risk Coverage

We check how many unique counterparty countries appear in the transaction data 
and what share successfully join to the country_risk table. This is critical 
because geographic risk is a core AML signal. Transactions routed through 
high-risk or FATF-listed countries must be captured reliably in our features.

In [10]:
# Unique counterparty countries in transactions
unique_countries = transactions['counterparty_bank_country'].dropna().unique()
print(f"Unique counterparty countries in transactions: {len(unique_countries)}")

# How many join to country_risk
risk_countries = set(country_risk['country_code'].unique())
matched = [c for c in unique_countries if c in risk_countries]
unmatched = [c for c in unique_countries if c not in risk_countries]

print(f"Matched to country_risk: {len(matched)}")
print(f"Unmatched: {len(unmatched)}")
if unmatched:
    print(f"Unmatched countries: {unmatched}")

print(f"\n=== FATF status distribution in country_risk ===")
print(country_risk['fatf_status'].value_counts())

print(f"\n=== EU high risk list ===")
print(country_risk['eu_high_risk_list'].value_counts())

Unique counterparty countries in transactions: 36
Matched to country_risk: 36
Unmatched: 0

=== FATF status distribution in country_risk ===
fatf_status
compliant              32
enhanced_monitoring    11
high_risk               4
blacklisted             1
Name: count, dtype: int64

=== EU high risk list ===
eu_high_risk_list
False    40
True      8
Name: count, dtype: int64


# 2. Feature Engineering

**Goal:** Transform 77,000 raw transactions into a single table with one row per 
customer and one column per AML signal — giving the model a complete behavioral 
profile for each of the 1,200 customers to learn from.

Each column is a feature that answers a specific compliance question, for example:
- `cash_transaction_ratio` - what share of this customer's transactions are cash?
- `structuring_count` - how many times did they transact just below the 15,000 DKK threshold?
- `high_risk_country_ratio` - what share of their transactions go to high-risk countries?

The output is this single feature table, ready for model training.

## 2.1 Cash Intensity

Cash transactions are the primary entry point for money laundering (placement stage).
We compute customer-level features capturing how much and how often a customer 
uses cash - and whether they show structuring patterns just below NordikBank's 
15,000 DKK internal reporting threshold.

The output is a summary table with one row per customer and one column per cash signal.
We print a statistical summary to sanity check the distributions before moving on -
for example, if the max cash_transaction_ratio is 85%, that outlier is exactly 
the kind of customer we are looking for.

In [11]:
# Work on approved transactions only for behavioral features
approved = transactions[transactions['status'] == 'approved'].copy()  # exclude declined transactions

# Flag cash transaction types
cash_types = ['cash_deposit', 'cash_withdrawal']  # define what counts as cash
approved['is_cash'] = approved['transaction_type'].isin(cash_types)  # True/False column per transaction

# Customer-level cash intensity features
cash_features = approved.groupby('customer_id').agg(
    total_transactions=('transaction_id', 'count'),  # total approved transactions per customer
    cash_transaction_count=('is_cash', 'sum'),  # number of cash transactions per customer
    cash_volume=('amount', lambda x: x[approved.loc[x.index, 'is_cash']].abs().sum()),  # total cash amount (absolute value)
    total_volume=('amount', lambda x: x.abs().sum()),  # total transaction volume (absolute value)
).reset_index()  # bring customer_id back as a column

# Cash ratio features
cash_features['cash_transaction_ratio'] = (
    cash_features['cash_transaction_count'] / cash_features['total_transactions']  # share of transactions that are cash
)
cash_features['cash_volume_ratio'] = (
    cash_features['cash_volume'] / cash_features['total_volume']  # share of total volume that is cash
)

# Structuring — transactions just below 15,000 DKK threshold
threshold = 15000  # NordikBank's internal reporting threshold
approved['is_structuring_candidate'] = (
    approved['is_cash'] &  # must be a cash transaction
    (approved['amount'].abs() >= threshold * 0.8) &  # at least 80% of threshold (12,000 DKK)
    (approved['amount'].abs() < threshold)  # strictly below threshold
)

# Count structuring candidates per customer
structuring = approved.groupby('customer_id')['is_structuring_candidate'].sum().reset_index()
structuring.columns = ['customer_id', 'structuring_count']  # rename for clarity

# Merge structuring count into cash features
cash_features = cash_features.merge(structuring, on='customer_id', how='left')  # left join to keep all customers

print(cash_features.describe().round(2))

       total_transactions  cash_transaction_count  cash_volume  total_volume  \
count             1200.00                 1200.00      1200.00       1200.00   
mean                63.17                    6.42     28153.52    3271648.50   
std                 28.10                   19.08    113688.60    9148163.69   
min                 18.00                    0.00         0.00      93641.16   
25%                 47.00                    1.00       200.00     384318.76   
50%                 58.00                    2.50      3500.00     588901.86   
75%                 70.00                    5.00      8000.00    1060290.50   
max                206.00                  138.00    794600.00   67916658.92   

       cash_transaction_ratio  cash_volume_ratio  structuring_count  
count                 1200.00            1200.00            1200.00  
mean                     0.08               0.03               0.46  
std                      0.12               0.11               2.01  

## 2.2 Spending Patterns

We capture how much and how frequently a customer transacts, and whether their 
behavior changed significantly between the first and second half of 2024.
A sudden spike in transaction volume or frequency, especially when inconsistent 
with their historical baseline, is a classic layering signal in AML.

The deviation features compare H1 2024 (Jan–Jun) against the pre-computed 
baselines (Jul–Dec 2024), capturing behavioral shifts over time.

**Relevant datasets:** `transactions.csv`, `baselines.csv`

In [12]:
# Split approved transactions into H1 and H2 2024
approved['month'] = approved['timestamp'].dt.to_period('M')  # extract month from timestamp
h1 = approved[approved['timestamp'].dt.month <= 6]  # Jan–Jun 2024
h2 = approved[approved['timestamp'].dt.month > 6]   # Jul–Dec 2024

# Customer-level spending features from full year
spending_features = approved.groupby('customer_id').agg(
    avg_transaction_amount=('amount', lambda x: x.abs().mean()),  # average transaction size
    std_transaction_amount=('amount', lambda x: x.abs().std()),   # variability in transaction size
    max_transaction_amount=('amount', lambda x: x.abs().max()),   # largest single transaction
    monthly_transaction_count=('transaction_id', 'count'),        # total transactions across year
).reset_index()

# Average monthly transaction count (divide by 12 months)
spending_features['avg_monthly_count'] = (
    spending_features['monthly_transaction_count'] / 12  # normalise to monthly average
)

# H1 aggregations
h1_agg = h1.groupby('customer_id').agg(
    h1_volume=('amount', lambda x: x.abs().sum()),   # total volume in H1
    h1_count=('transaction_id', 'count'),             # transaction count in H1
).reset_index()

# H2 aggregations
h2_agg = h2.groupby('customer_id').agg(
    h2_volume=('amount', lambda x: x.abs().sum()),   # total volume in H2
    h2_count=('transaction_id', 'count'),             # transaction count in H2
).reset_index()

# Merge H1 and H2 together
deviation = h1_agg.merge(h2_agg, on='customer_id', how='outer')  # outer join to keep all customers

# Compute deviation features — how much did behavior change from H1 to H2?
deviation['volume_change_ratio'] = (
    (deviation['h2_volume'] - deviation['h1_volume']) / deviation['h1_volume']  # % change in volume
)
deviation['count_change_ratio'] = (
    (deviation['h2_count'] - deviation['h1_count']) / deviation['h1_count']  # % change in transaction count
)

# Merge spending features with deviation features
spending_features = spending_features.merge(
    deviation[['customer_id', 'volume_change_ratio', 'count_change_ratio']], 
    on='customer_id', 
    how='left'  # keep all customers
)

print(spending_features.describe().round(2))

       avg_transaction_amount  std_transaction_amount  max_transaction_amount  \
count                 1200.00                 1200.00                 1200.00   
mean                 50631.50                88127.67               361952.79   
std                 146042.02               257708.83              1101738.23   
min                   2360.30                 2135.37                 4742.36   
25%                   6951.98                 9697.39                26150.16   
50%                  10240.05                14710.18                38220.74   
75%                  17137.03                24688.16                73759.31   
max                1246641.28              2090410.19              7328554.68   

       monthly_transaction_count  avg_monthly_count  volume_change_ratio  \
count                    1200.00            1200.00              1199.00   
mean                       63.17               5.26                53.07   
std                        28.10          

## 2.3 Counterparty Networks

Money laundering requires moving funds between entities. We capture how many 
unique counterparties a customer transacts with, how concentrated their activity 
is around a single counterparty, and whether they transact with many one-time 
counterparties; a signal of layering.

**Relevant datasets:** `transactions.csv`

In [13]:
# Counterparty features — approved transactions only
# Only keep transactions where counterparty_id is known
counterparty_txn = approved[approved['counterparty_id'].notna()].copy()  # exclude transactions with no counterparty

# Customer-level counterparty network features
counterparty_features = counterparty_txn.groupby('customer_id').agg(
    num_unique_counterparties=('counterparty_id', 'nunique'),  # how many distinct counterparties
    total_counterparty_transactions=('transaction_id', 'count'),  # total transactions with known counterparty
).reset_index()

# Counterparty concentration — what share of transactions go to the top counterparty?
top_counterparty = (
    counterparty_txn.groupby(['customer_id', 'counterparty_id'])
    .size()  # count transactions per customer-counterparty pair
    .reset_index(name='pair_count')
)
top_counterparty = top_counterparty.sort_values('pair_count', ascending=False)
top_counterparty = top_counterparty.groupby('customer_id').first().reset_index()  # keep top counterparty per customer
top_counterparty['top_counterparty_share'] = (
    top_counterparty['pair_count'] / counterparty_features.set_index('customer_id')
    .loc[top_counterparty['customer_id'], 'total_counterparty_transactions'].values  # share of transactions to top counterparty
)

# One-time counterparties — counterparties seen only once (layering signal)
one_time = (
    counterparty_txn.groupby(['customer_id', 'counterparty_id'])
    .size()
    .reset_index(name='pair_count')
)
one_time['is_one_time'] = one_time['pair_count'] == 1  # flag counterparties seen exactly once
one_time_count = one_time.groupby('customer_id')['is_one_time'].sum().reset_index()
one_time_count.columns = ['customer_id', 'one_time_counterparty_count']  # rename for clarity

# Merge all counterparty features together
counterparty_features = counterparty_features.merge(
    top_counterparty[['customer_id', 'top_counterparty_share']], 
    on='customer_id', how='left'
)
counterparty_features = counterparty_features.merge(
    one_time_count, on='customer_id', how='left'
)

# Merge customers with no counterparty transactions — fill with 0
counterparty_features = customers[['customer_id']].merge(
    counterparty_features, on='customer_id', how='left'
).fillna(0)  # customers with no known counterparties get 0 — documented imputation

print(counterparty_features.describe().round(2))

       num_unique_counterparties  total_counterparty_transactions  \
count                    1200.00                          1200.00   
mean                       10.76                            44.83   
std                         5.87                            20.85   
min                         3.00                            12.00   
25%                         7.00                            32.00   
50%                         9.00                            43.00   
75%                        12.00                            53.00   
max                        35.00                           182.00   

       top_counterparty_share  one_time_counterparty_count  
count                 1200.00                      1200.00  
mean                     0.32                         4.65  
std                      0.13                         3.54  
min                      0.09                         0.00  
25%                      0.23                         2.00  
50%         

## 2.4 Geographic Spread

Transactions routed through high-risk or FATF-listed countries are a core AML signal.
We compute customer-level exposure to risky jurisdictions by joining counterparty 
bank countries to the country risk table - capturing not just whether a customer 
transacts internationally, but how risky those destinations are.

**Relevant datasets:** `transactions.csv`, `country_risk.csv`

In [14]:
# Geographic features — join counterparty country to country risk table
intl_txn = approved[approved['counterparty_bank_country'].notna()].copy()  # only transactions with a known country

# Join country risk to international transactions
intl_txn = intl_txn.merge(
    country_risk[['country_code', 'fatf_status', 'eu_high_risk_list', 'corruption_perception_index']],
    left_on='counterparty_bank_country',
    right_on='country_code',
    how='left'  # keep all international transactions even if country not in risk table
)

# Flag high risk transactions
intl_txn['is_high_risk_country'] = (
    intl_txn['fatf_status'].isin(['high_risk', 'blacklisted']) |  # FATF high risk or blacklisted
    intl_txn['eu_high_risk_list'] == True  # on EU high risk list
)

# Customer-level geographic features
geo_features = intl_txn.groupby('customer_id').agg(
    num_international_transactions=('transaction_id', 'count'),  # total international transactions
    num_high_risk_country_transactions=('is_high_risk_country', 'sum'),  # transactions to high risk countries
    num_unique_countries=('counterparty_bank_country', 'nunique'),  # number of distinct countries
    avg_corruption_index=('corruption_perception_index', 'mean'),  # average CPI of destination countries
).reset_index()

# Ratio of high risk country transactions
geo_features['high_risk_country_ratio'] = (
    geo_features['num_high_risk_country_transactions'] /
    geo_features['num_international_transactions']  # share of international transactions to high risk countries
)

# Merge back to all customers — fill 0 for customers with no international transactions
geo_features = customers[['customer_id']].merge(
    geo_features, on='customer_id', how='left'
).fillna(0)  # customers with no international transactions get 0 — documented imputation

print(geo_features.describe().round(2))

       num_international_transactions  num_high_risk_country_transactions  \
count                         1200.00                             1200.00   
mean                             2.12                                0.46   
std                              5.97                                2.75   
min                              0.00                                0.00   
25%                              0.00                                0.00   
50%                              0.00                                0.00   
75%                              1.00                                0.00   
max                             72.00                               67.00   

       num_unique_countries  avg_corruption_index  high_risk_country_ratio  
count               1200.00               1200.00                  1200.00  
mean                   1.06                 23.60                     0.08  
std                    1.78                 28.97                     0.24 

## 2.5 Income Consistency

A fundamental AML signal is when a customer's actual transaction volume 
significantly exceeds their declared income or turnover. This suggests 
undeclared funds entering the financial system - a classic placement indicator.

We compute the ratio of total inflow to declared income for personal customers, 
and total inflow to declared turnover for corporate and sole trader customers.
Note that corporate and SME customers have no declared income - we handle 
each customer type separately to avoid silent nulls.

**Relevant datasets:** `transactions.csv`, `customers.csv`

In [15]:
# Compute total annual inflow per customer (approved transactions only)
inflow = approved[approved['amount'] > 0].groupby('customer_id').agg(
    total_annual_inflow=('amount', 'sum')  # sum of all positive amounts (money coming in)
).reset_index()

# Merge inflow with customer declared income/turnover
income_features = customers[['customer_id', 'customer_type', 
                              'declared_annual_income', 
                              'declared_annual_turnover']].merge(
    inflow, on='customer_id', how='left'  # keep all customers even if no inflow
)

# Personal customers — compare inflow to declared annual income
personal_mask = income_features['customer_type'] == 'personal'
income_features.loc[personal_mask, 'income_to_inflow_ratio'] = (
    income_features.loc[personal_mask, 'total_annual_inflow'] /
    income_features.loc[personal_mask, 'declared_annual_income']  # how many times inflow exceeds declared income
)

# Sole traders — have both income and turnover, use turnover as primary benchmark
sole_trader_mask = income_features['customer_type'] == 'sole_trader'
income_features.loc[sole_trader_mask, 'income_to_inflow_ratio'] = (
    income_features.loc[sole_trader_mask, 'total_annual_inflow'] /
    income_features.loc[sole_trader_mask, 'declared_annual_turnover']  # compare inflow to declared turnover
)

# Corporate and SME — compare inflow to declared annual turnover
corp_mask = income_features['customer_type'].isin(['corporate', 'SME'])
income_features.loc[corp_mask, 'income_to_inflow_ratio'] = (
    income_features.loc[corp_mask, 'total_annual_inflow'] /
    income_features.loc[corp_mask, 'declared_annual_turnover']  # compare inflow to declared turnover
)

# Keep only relevant columns
income_features = income_features[['customer_id', 'total_annual_inflow', 
                                    'income_to_inflow_ratio']]

print(income_features.describe().round(2))
print(f"\nNull income_to_inflow_ratio: {income_features['income_to_inflow_ratio'].isna().sum()}")

       total_annual_inflow  income_to_inflow_ratio
count              1200.00                 1142.00
mean            3070238.63                    1.33
std             9124772.22                    2.65
min               55976.94                    0.02
25%              299279.00                    0.99
50%              469113.98                    1.00
75%              847417.47                    1.01
max            67840551.26                   43.40

Null income_to_inflow_ratio: 58


## 2.6 Declined Transaction Features

A high rate of declined transactions is an AML signal - it can indicate velocity 
testing or structuring attempts, where a criminal probes the system to find 
the maximum amount that gets approved without triggering alerts.

**Relevant datasets:** `transactions.csv`

In [16]:
# Declined transaction features — use full transaction dataset (not approved only)
declined = transactions[transactions['status'] == 'declined'].copy()  # isolate declined transactions

# Total attempted transactions per customer (approved + declined)
total_attempted = transactions.groupby('customer_id').agg(
    total_attempted=('transaction_id', 'count')  # all transactions regardless of status
).reset_index()

# Declined transaction count and volume per customer
declined_features = declined.groupby('customer_id').agg(
    declined_count=('transaction_id', 'count'),  # number of declined transactions
    declined_volume=('amount', lambda x: x.abs().sum()),  # total attempted volume that was declined
    avg_declined_amount=('amount', lambda x: x.abs().mean()),  # average size of declined transactions
).reset_index()

# Merge with total attempted to compute decline rate
declined_features = total_attempted.merge(
    declined_features, on='customer_id', how='left'
)

# Fill customers with no declined transactions with 0
declined_features['declined_count'] = declined_features['declined_count'].fillna(0)  # no declines = 0
declined_features['declined_volume'] = declined_features['declined_volume'].fillna(0)  # no declines = 0
declined_features['avg_declined_amount'] = declined_features['avg_declined_amount'].fillna(0)  # no declines = 0

# Decline rate — share of attempted transactions that were declined
declined_features['decline_rate'] = (
    declined_features['declined_count'] / declined_features['total_attempted']  # proportion of failed transactions
)

# Keep only relevant columns
declined_features = declined_features[['customer_id', 'declined_count', 
                                        'declined_volume', 'avg_declined_amount', 
                                        'decline_rate']]

print(declined_features.describe().round(2))
print(f"\nCustomers with at least one declined transaction: {(declined_features['declined_count'] > 0).sum()}")

       declined_count  declined_volume  avg_declined_amount  decline_rate
count         1200.00          1200.00              1200.00       1200.00
mean             1.31         33518.90             19564.93          0.02
std              1.35        305453.56            180866.67          0.02
min              0.00             0.00                 0.00          0.00
25%              0.00             0.00                 0.00          0.00
50%              1.00           353.42               253.51          0.02
75%              2.00          1353.20               663.61          0.03
max              9.00       5977842.95           3211563.50          0.13

Customers with at least one declined transaction: 806


## 2.7 Customer & Account Attributes

We enrich the feature table with static customer attributes and account-level 
features. These capture risk indicators that are not transaction-based, such as 
PEP status, sanctions flags, and KYC risk rating, as well as account structure 
signals like whether a personal customer holds a business account.

**Relevant datasets:** `customers.csv`, `accounts.csv`

In [17]:
# Static customer-level risk attributes
customer_attributes = customers[['customer_id', 'customer_type', 'kyc_risk_rating',
                                  'pep_status', 'sanctions_screening_flag',
                                  'age', 'nationality', 'residency_country',
                                  'registration_date']].copy()

# Encode KYC risk rating as ordinal — low=0, medium=1, high=2
kyc_map = {'low': 0, 'medium': 1, 'high': 2}
customer_attributes['kyc_risk_score'] = (
    customer_attributes['kyc_risk_rating'].map(kyc_map)  # convert text rating to numeric
)

# Encode customer type as dummy variables
customer_attributes = pd.get_dummies(
    customer_attributes, 
    columns=['customer_type'],  # one column per customer type
    prefix='is'  # e.g. is_personal, is_corporate
)

# Join nationality and residency to country risk
customer_attributes = customer_attributes.merge(
    country_risk[['country_code', 'fatf_status', 'eu_high_risk_list']].rename(
        columns={'country_code': 'nationality', 
                 'fatf_status': 'nationality_fatf', 
                 'eu_high_risk_list': 'nationality_eu_risk'}),
    on='nationality', how='left'  # enrich nationality with risk status
)

customer_attributes = customer_attributes.merge(
    country_risk[['country_code', 'fatf_status', 'eu_high_risk_list']].rename(
        columns={'country_code': 'residency_country',
                 'fatf_status': 'residency_fatf',
                 'eu_high_risk_list': 'residency_eu_risk'}),
    on='residency_country', how='left'  # enrich residency with risk status
)

# Flag high risk nationality or residency
customer_attributes['high_risk_nationality'] = (
    customer_attributes['nationality_fatf'].isin(['high_risk', 'blacklisted']) |
    customer_attributes['nationality_eu_risk'] == True  # nationality in high risk jurisdiction
)

customer_attributes['high_risk_residency'] = (
    customer_attributes['residency_fatf'].isin(['high_risk', 'blacklisted']) |
    customer_attributes['residency_eu_risk'] == True  # residency in high risk jurisdiction
)

# Account-level features
# Flag personal customers holding a business account
account_types = accounts.groupby('customer_id')['account_type'].apply(list).reset_index()
account_types['has_business_account'] = account_types['account_type'].apply(
    lambda x: 'business' in x  # True if customer holds any business account
)
account_types['num_accounts'] = account_types['account_type'].apply(len)  # total number of accounts

# Average monthly balance and flow features from accounts
account_features = accounts.groupby('customer_id').agg(
    avg_balance=('avg_monthly_balance_6m', 'mean'),        # average balance across all accounts
    avg_inflow=('avg_monthly_inflow_6m', 'mean'),          # average monthly inflow
    avg_outflow=('avg_monthly_outflow_6m', 'mean'),        # average monthly outflow
).reset_index()

# Merge account features together
account_features = account_features.merge(
    account_types[['customer_id', 'has_business_account', 'num_accounts']], 
    on='customer_id', how='left'
)

print("=== Customer attributes shape ===")
print(customer_attributes.shape)
print("\n=== PEP status distribution ===")
print(customer_attributes['pep_status'].value_counts())
print("\n=== Sanctions flag distribution ===")
print(customer_attributes['sanctions_screening_flag'].value_counts())
print("\n=== KYC risk score distribution ===")
print(customer_attributes['kyc_risk_score'].value_counts().sort_index())
print("\n=== Account features ===")
print(account_features.describe().round(2))
print(f"\nPersonal customers with business account: {account_types[account_types['has_business_account']].shape[0]}")

=== Customer attributes shape ===
(1200, 19)

=== PEP status distribution ===
pep_status
False    1192
True        8
Name: count, dtype: int64

=== Sanctions flag distribution ===
sanctions_screening_flag
False    1198
True        2
Name: count, dtype: int64

=== KYC risk score distribution ===
kyc_risk_score
0    1113
1      79
2       8
Name: count, dtype: int64

=== Account features ===
       avg_balance  avg_inflow  avg_outflow  num_accounts
count      1200.00     1200.00      1200.00       1200.00
mean     189249.51    89979.04      9003.07          2.25
std      551550.01   240742.22     26453.32          1.06
min         739.15     2463.84       126.54          1.00
25%       20990.13    12830.64      1204.23          1.00
50%       36681.60    20592.59      4252.65          2.00
75%       78001.19    41789.47      7593.95          3.00
max     5276294.66  2294657.96    319880.79          5.00

Personal customers with business account: 257


## 2.8 Baseline Integration

The baselines.csv file contains pre-computed behavioral features covering 
Jul–Dec 2024. Rather than recomputing these, we integrate them directly 
into our feature table. We also compute deviation features comparing 
H1 2024 (Jan–Jun) behavior against the baseline period - capturing 
whether a customer's behavior changed significantly over the year.

**Relevant datasets:** `baselines.csv`

In [18]:
# Baselines are already at customer grain — one row per customer
# Inspect the columns available
print("=== Baseline columns ===")
print(baselines.columns.tolist())
print("\n=== Baseline sample ===")
print(baselines.describe().round(2))

=== Baseline columns ===
['customer_id', 'avg_monthly_transaction_count', 'avg_monthly_volume', 'max_single_transaction_6m', 'pct_international_transactions', 'pct_cash_transactions', 'num_unique_counterparties_6m', 'transaction_time_entropy', 'geographic_spread_score', 'dormancy_periods_count']

=== Baseline sample ===
       avg_monthly_transaction_count  avg_monthly_volume  \
count                        1200.00             1200.00   
mean                            5.45           275155.84   
std                             2.41           760168.88   
min                             1.60             6386.42   
25%                             4.20            31608.60   
50%                             5.00            48331.37   
75%                             6.00            92683.70   
max                            18.00          5248226.16   

       max_single_transaction_6m  pct_international_transactions  \
count                    1200.00                         1200.00   
m

Merge

In [19]:
# Baselines are already at customer grain — merge directly
# No transformation needed, just integrate as-is
baseline_features = baselines.copy()  # keep all baseline columns

# Compute deviation features — compare our H1 features to baseline (H2)
# We already have h1_agg from section 2.2 — merge with baselines to compute deviation

# Volume deviation — how much did monthly volume change from H1 to H2?
h1_monthly_volume = h1_agg.copy()
h1_monthly_volume['h1_avg_monthly_volume'] = (
    h1_monthly_volume['h1_volume'] / 6  # average monthly volume in H1 (6 months)
)

deviation_features = h1_monthly_volume[['customer_id', 'h1_avg_monthly_volume']].merge(
    baseline_features[['customer_id', 'avg_monthly_volume']], 
    on='customer_id', how='left'
)

# Volume deviation ratio — how much did monthly volume change relative to H1?
deviation_features['volume_deviation_ratio'] = (
    (deviation_features['avg_monthly_volume'] - deviation_features['h1_avg_monthly_volume']) /
    deviation_features['h1_avg_monthly_volume']  # % change in monthly volume from H1 to H2
)

# Transaction count deviation
h1_monthly_count = h1_agg.copy()
h1_monthly_count['h1_avg_monthly_count'] = (
    h1_monthly_count['h1_count'] / 6  # average monthly count in H1
)

deviation_features = deviation_features.merge(
    h1_monthly_count[['customer_id', 'h1_avg_monthly_count']], 
    on='customer_id', how='left'
)
deviation_features = deviation_features.merge(
    baseline_features[['customer_id', 'avg_monthly_transaction_count']], 
    on='customer_id', how='left'
)

deviation_features['count_deviation_ratio'] = (
    (deviation_features['avg_monthly_transaction_count'] - deviation_features['h1_avg_monthly_count']) /
    deviation_features['h1_avg_monthly_count']  # % change in monthly transaction count from H1 to H2
)

print("=== Deviation features ===")
print(deviation_features[['volume_deviation_ratio', 'count_deviation_ratio']].describe().round(2))
print(f"\nNull volume deviation: {deviation_features['volume_deviation_ratio'].isna().sum()}")
print(f"Null count deviation: {deviation_features['count_deviation_ratio'].isna().sum()}")

=== Deviation features ===
       volume_deviation_ratio  count_deviation_ratio
count                 1199.00                1199.00
mean                    63.04                   0.23
std                   1402.47                   2.55
min                     -0.98                  -0.57
25%                     -0.10                  -0.09
50%                      0.01                   0.04
75%                      0.14                   0.20
max                  36564.46                  63.20

Null volume deviation: 0
Null count deviation: 0


## 2.9 Consolidation

We merge all feature groups into a single table at customer grain — one row 
per customer, one column per feature. This is the final feature table that 
will be handed off for model training. We document all null handling explicitly 
and save the output as a versioned parquet file.

**Relevant datasets:** All feature tables built in sections 2.1–2.8

In [20]:
# Start with all customers as the base — ensures all 1200 customers are present
feature_table = customers[['customer_id', 'split', 'suspicious_activity_confirmed']].copy()

# Merge all feature groups one by one
feature_table = feature_table.merge(cash_features, on='customer_id', how='left')
feature_table = feature_table.merge(spending_features, on='customer_id', how='left')
feature_table = feature_table.merge(counterparty_features, on='customer_id', how='left')
feature_table = feature_table.merge(geo_features, on='customer_id', how='left')
feature_table = feature_table.merge(income_features, on='customer_id', how='left')
feature_table = feature_table.merge(declined_features, on='customer_id', how='left')
feature_table = feature_table.merge(
    customer_attributes.drop(columns=['kyc_risk_rating', 'nationality', 
                                       'residency_country', 'registration_date',
                                       'nationality_fatf', 'nationality_eu_risk',
                                       'residency_fatf', 'residency_eu_risk']), 
    on='customer_id', how='left'
)
feature_table = feature_table.merge(account_features, on='customer_id', how='left')
feature_table = feature_table.merge(baseline_features, on='customer_id', how='left')
feature_table = feature_table.merge(
    deviation_features[['customer_id', 'volume_deviation_ratio', 'count_deviation_ratio']], 
    on='customer_id', how='left'
)

# Document all null handling
print("=== Null counts before imputation ===")
nulls = feature_table.isnull().sum()
print(nulls[nulls > 0])

# Impute remaining nulls
# Deviation features — 1 customer with no H1 transactions, fill with 0 (no change observed)
feature_table['volume_deviation_ratio'] = feature_table['volume_deviation_ratio'].fillna(0)
feature_table['count_deviation_ratio'] = feature_table['count_deviation_ratio'].fillna(0)

# Income to inflow ratio — 58 nulls from customers with missing declared income/turnover
# Fill with 1.0 — neutral value meaning inflow equals declared amount (conservative assumption)
feature_table['income_to_inflow_ratio'] = feature_table['income_to_inflow_ratio'].fillna(1.0)

# Geographic features — customers with no international transactions already filled with 0 in 2.4
# Corruption index — fill with 0 for customers with no international transactions
feature_table['avg_corruption_index'] = feature_table['avg_corruption_index'].fillna(0)

print("\n=== Null counts after imputation ===")
nulls_after = feature_table.isnull().sum()
print(nulls_after[nulls_after > 0])

print(f"\n=== Final feature table shape ===")
print(feature_table.shape)
print(f"\nColumns: {feature_table.columns.tolist()}")

=== Null counts before imputation ===
suspicious_activity_confirmed    500
volume_change_ratio                1
count_change_ratio                 1
income_to_inflow_ratio            58
age                              104
volume_deviation_ratio             1
count_deviation_ratio              1
dtype: int64

=== Null counts after imputation ===
suspicious_activity_confirmed    500
volume_change_ratio                1
count_change_ratio                 1
age                              104
dtype: int64

=== Final feature table shape ===
(1200, 58)

Columns: ['customer_id', 'split', 'suspicious_activity_confirmed', 'total_transactions', 'cash_transaction_count', 'cash_volume', 'total_volume', 'cash_transaction_ratio', 'cash_volume_ratio', 'structuring_count', 'avg_transaction_amount', 'std_transaction_amount', 'max_transaction_amount', 'monthly_transaction_count', 'avg_monthly_count', 'volume_change_ratio', 'count_change_ratio', 'num_unique_counterparties', 'total_counterparty_transact

Nulls are handled as follows:
- `volume_change_ratio` and `count_change_ratio` — filled with 0, one customer had no H1 activity so no change can be observed
- `income_to_inflow_ratio` — filled with 1.0, a neutral value assuming inflow matches declared income where data is missing
- `age` — filled with -1 as a sentinel value for corporate and SME customers where age is not applicable by design
- `suspicious_activity_confirmed` — 500 nulls remain intentionally, these are the test rows withheld by the panel

The table is saved as `features_v1.parquet` — parquet is preferred over CSV 
as it preserves data types and is faster to load for model training.

**Relevant datasets:** All feature tables built in sections 2.1–2.8

In [27]:
# Fix remaining nulls
feature_table['volume_change_ratio'] = feature_table['volume_change_ratio'].fillna(0)  # no H1 activity means no change observed
feature_table['count_change_ratio'] = feature_table['count_change_ratio'].fillna(0)  # same logic as volume
feature_table['age'] = feature_table['age'].fillna(-1)  # -1 sentinel for corporate/SME customers where age is not applicable by design

# Final null check — only suspicious_activity_confirmed should remain (test rows, expected)
print("=== Null counts after all imputation ===")
nulls_final = feature_table.isnull().sum()
remaining = nulls_final[nulls_final > 0]
print(remaining if len(remaining) > 0 else "No nulls except suspicious_activity_confirmed (expected)")

# Save as CSV instead of parquet due to Python 3.14 / pyarrow compatibility issue
save_path = '../../data/features_base.csv'  # shared data folder accessible to all team members
feature_table.to_csv(save_path, index=False)  # save without row index
print(f"Saved to {save_path}")
print(f"\nFinal shape: {feature_table.shape}")

=== Null counts after all imputation ===
suspicious_activity_confirmed    500
dtype: int64
Saved to ../../data/features_base.csv

Final shape: (1200, 58)


In [28]:
!pip install tsfresh


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2.10 tsfresh Feature Extraction

We apply tsfresh to extract time-series features from the approved transaction 
history of each customer. tsfresh automatically computes hundreds of features 
capturing the time-dimension of behavior — such as transaction amount volatility, 
trend, and acceleration — that manual aggregations cannot capture as elegantly.

We run tsfresh on absolute transaction amounts, sorted by timestamp, on approved 
transactions only. The resulting features are merged with the base feature table 
and saved as `features_base_and_tsfresh.csv`.

Srijita will apply `RelevantFeatureAugmenter` on the training set to filter down 
to only statistically significant features before model training — avoiding 
data leakage from the val and test sets.

**Relevant datasets:** `transactions.csv`

In [29]:
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import impute

# Prepare transaction data for tsfresh
# tsfresh needs: entity id (customer_id), time column (timestamp), value columns
tsfresh_input = approved[['customer_id', 'timestamp', 'amount']].copy()  # approved transactions only
tsfresh_input['amount'] = tsfresh_input['amount'].abs()  # use absolute values — no signed amounts
tsfresh_input['timestamp'] = tsfresh_input['timestamp'].astype(int)  # tsfresh needs numeric time

# Extract features — this may take a few minutes
print("Extracting tsfresh features — this may take a few minutes...")
tsfresh_features = extract_features(
    tsfresh_input,
    column_id='customer_id',    # one profile per customer
    column_sort='timestamp',    # sort by time
    column_value='amount',      # extract features from transaction amounts
    disable_progressbar=False   # show progress
)

# Impute any nulls tsfresh generates internally
impute(tsfresh_features)

print(f"\ntsfresh generated {tsfresh_features.shape[1]} features for {tsfresh_features.shape[0]} customers")

Extracting tsfresh features — this may take a few minutes...


Feature Extraction: 100%|██████████| 40/40 [00:28<00:00,  1.42it/s]
c:\Users\jasmi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tsfresh\utilities\dataframe_functions.py:198: RuntimeWarning: The columns ['amount__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(



tsfresh generated 783 features for 1200 customers


### 2.10.1 Merging tsfresh Features with Base Feature Table

We merge the 783 tsfresh-generated features with our base feature table,
producing a combined dataset. This combined table is saved as 
`features_base_and_tsfresh.csv` and handed off to Srijita for modeling.

In [30]:
# Reset index so customer_id becomes a column
tsfresh_features = tsfresh_features.reset_index().rename(columns={'index': 'customer_id'})  # bring customer_id back as column

# Merge tsfresh features with base feature table
feature_table_tsfresh = feature_table.merge(
    tsfresh_features, 
    on='customer_id', 
    how='left'  # keep all 1200 customers
)

print(f"=== Combined feature table shape ===")
print(f"Rows: {feature_table_tsfresh.shape[0]}")
print(f"Columns: {feature_table_tsfresh.shape[1]}")
print(f"Base features: 58")
print(f"tsfresh features added: {feature_table_tsfresh.shape[1] - 58}")

# Save combined feature table
save_path_tsfresh = '../../data/features_base_and_tsfresh.csv'
feature_table_tsfresh.to_csv(save_path_tsfresh, index=False)
print(f"\nSaved to {save_path_tsfresh}")

=== Combined feature table shape ===
Rows: 1200
Columns: 841
Base features: 58
tsfresh features added: 783

Saved to ../../data/features_base_and_tsfresh.csv
